<!--nav--> [🗺 Learning path](README.md) · **1/50** · [Simple Multi-GPU Training](./Simple_MultiGPU_Training.ipynb) ▶

# Start Here: One Composable Stack

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Start_Here_One_Composable_Stack.ipynb)

There are forty-nine other notebooks in this repository and that is too many to start with.

This one is the entry point. It does not summarize the others — a summary of forty-nine
notebooks is a worse notebook. Instead it uses the thing they have in common: every one of them
contains a **model**, a piece of arithmetic that turns a configuration into a prediction. Those
models are now a library, `servingkit`, and this notebook composes them.

The shape of what follows:

1. **Declare a workload.** Chat, agent, or offline batch. Three lines.
2. **See which optimizations apply to it** — and, more usefully, which do not and why.
3. **Get them ordered by money**, one at a time, with the running cost at each rung.
4. **See which pairs fight**, with the reason predicted rather than discovered.
5. **Run the kernel** behind any lever, on a CPU, in about a minute.
6. **Check that none of it drifted** from the notebooks it came from.

Then go read the notebook for whichever lever turned out to matter for you. The plan tells you
which one that is, which is the only reason to have forty-nine of them.

### What makes this composable rather than a pile of functions

A lever is not just a function. It carries four declarations:

| declaration | what it buys |
|---|---|
| `domain` | two levers in the same domain are **alternatives** — a model has one weight format |
| `spends` | two levers spending the same resource will **fight**, and the table says so in advance |
| `requires` | a workload lacking the property is **not offered** the lever, rather than shown a weak number |
| `kernel` | "read the code" has an address: a compilable file in [`kernels/`](https://github.com/sugeerth/gpu-training-notebooks/tree/main/kernels) |

That last column is the point of the whole repository. These are not abstractions over vendor
documentation; each one is a program that runs.

In [1]:
# Setup. On Colab this clones the repo and installs the package; locally it just imports it.
import subprocess, sys
from pathlib import Path

def find_repo():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "servingkit" / "__init__.py").exists():
            return cand
    return None

URL = "https://github.com/sugeerth/gpu-training-notebooks"
BRANCH = "claude/serving-optimization-notebooks-jyerty"   # until this lands on main

REPO = find_repo()
if REPO is None:
    dest = Path("/content/gpu-training-notebooks")
    if not (dest / "servingkit").exists():
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", URL, str(dest)], check=True)
        if not (dest / "servingkit").exists():
            subprocess.run(["git", "fetch", "--depth", "1", "origin", BRANCH], cwd=str(dest))
            subprocess.run(["git", "checkout", "FETCH_HEAD"], cwd=str(dest))
    REPO = dest
sys.path.insert(0, str(REPO))

import servingkit as sk
print("servingkit", sk.__version__, "from", REPO)
print(f"{len(sk.LEVERS)} levers, {len(sk.available_kernels())} kernels, "
      f"{len(sk.MODELS)} models, {len(sk.GPUS)} accelerators")
print("\nno dependencies — this package is arithmetic over dicts, so it imports in a second")
print("and runs in CI without a GPU")

servingkit 0.1.0 from /home/user/gpu-training-notebooks
17 levers, 22 kernels, 8 models, 12 accelerators

no dependencies — this package is arithmetic over dicts, so it imports in a second
and runs in CI without a GPU


## 1 · Declare a workload

This is the step the repository was missing. Every notebook implicitly assumed a workload —
chat in most of them, an agent loop in one — and the reader had to work out which advice
transferred. Saying it out loud is what lets everything downstream be specific.

The properties are **derived**, not declared, so they cannot drift from the numbers. Change
`turns` and the raggedness appears; set `fanout=1` and the shared prefix disappears, because
there are no branches to share one.

In [2]:
agent = sk.Workload.agent(
    turns=30,                   # a long-running agent: thirty tool calls
    tool_result_tokens=2000,    # each result is a page of search output or a file
    tool_latency_s=2.0,         # and takes two seconds to come back
    fanout=4,                   # a planner spawning four sub-agents
    replayed=True,              # runs are re-run in an eval harness
)
chat = sk.Workload.chat(ctx=2048, batch=32)
offline = sk.Workload.batch(ctx=8192, batch=256)

for w in (chat, agent, offline):
    print(w.describe())
    print(f"  bottleneck: {w.bottleneck()} · {w.spare_compute():.0%} of math capacity idle\n")

print("The agent's context is derived from its turn count, not passed — which is the whole")
print("reason its bottleneck differs from the chat's.")

chat: Llama-3.1-8B, ctx 2,048, batch 32
  properties: long_context, spare_compute, weight_bound
  bottleneck: GEMMs · 86% of math capacity idle

agent: Llama-3.1-8B, ctx 66,050, batch 32
  properties: forks, idle_holds, long_context, mid_batch_prefill, predictable_output, ragged_batch, repeated_context, replayed, shared_prefix, spare_compute, structured_output
  bottleneck: attention · 86% of math capacity idle

batch: Llama-3.1-8B, ctx 8,192, batch 256
  properties: long_context, ragged_batch
  bottleneck: attention · 0% of math capacity idle

The agent's context is derived from its turn count, not passed — which is the whole
reason its bottleneck differs from the chat's.


## 2 · Which levers apply, and which do not

The second column is the interesting one. An `x` does not mean "weak here" — it means
**inapplicable**: the property the lever exploits is absent, so the number it would report is
meaningless rather than small.

In [3]:
print(sk.lever_table(agent.properties))

   lever                      domain               spends                   requires                      kernel
----------------------------------------------------------------------------------------------------------------
   int4 weights               weight_format        accuracy, spare_compute  any                           05_dequant_gemv.cu
   fp8 weights                weight_format        accuracy, spare_compute  any                           09_fp8_scaling.cu
   fp8 KV cache               kv_format            accuracy, memory         long_context                  09_fp8_scaling.cu
   KV eviction                kv_extent            accuracy, memory         long_context                  -
 x 2x batch                   batch                spare_compute, memory    spare_compute, weight_bound   -
 x speculation                tokens_per_step      spare_compute            spare_compute, weight_bound   18_spec_verify.cu
   TP=2                       topology             interconne

Two of those refusals are worth dwelling on, because they are where conventional advice is
wrong and the package is right.

**"Just raise the batch size"** has a precondition. Batching amortizes *one weight read* over
more sequences, so it needs the weight read to be the thing that hurts. At 66k of context
attention is seventeen times the GEMM, so doubling the batch doubles attention and buys nothing.

**Speculation** has the same precondition and a sharper failure: it multiplies the attention
term by `k+1`. Offered to a long-context agent it is not a weak optimization, it is a
regression. The first draft of this library gated it on `batch < 128` and cheerfully
recommended it to exactly the workload it would have harmed.

In [4]:
# The same two levers, on a workload that can actually use them.
print(sk.lever_table(chat.properties))
print()
for lv in ("2x batch", "speculation"):
    print(f"{lv}:")
    for w, label in ((chat, "chat 2k/32"), (agent, "agent 30 turns")):
        ok = sk.LEVERS[lv].applies_to(w.properties)
        missing = sk.LEVERS[lv].missing(w.properties)
        print(f"  {label:<16} {'applies' if ok else 'needs ' + ', '.join(missing)}")

   lever                      domain               spends                   requires                      kernel
----------------------------------------------------------------------------------------------------------------
   int4 weights               weight_format        accuracy, spare_compute  any                           05_dequant_gemv.cu
   fp8 weights                weight_format        accuracy, spare_compute  any                           09_fp8_scaling.cu
   fp8 KV cache               kv_format            accuracy, memory         long_context                  09_fp8_scaling.cu
   KV eviction                kv_extent            accuracy, memory         long_context                  -
   2x batch                   batch                spare_compute, memory    spare_compute, weight_bound   -
   speculation                tokens_per_step      spare_compute            spare_compute, weight_bound   18_spec_verify.cu
   TP=2                       topology             interconne

## 3 · The ladder: ordered by effort, costed in money

Levers added one at a time, in the order a team would reach for them — config changes first,
then the kernel swaps that cost nothing, then the ones that spend accuracy.

The `$` column includes the GPUs a lever rents, not just the tokens it moves. That matters:
a lever that halves your step time and doubles your bill has not obviously helped, and a planner
that reports only latency cannot tell you so.

In [5]:
stack = sk.recommend(agent)
print(sk.ladder_table(stack.ladder()))

configuration                      tok/s     prefill     wall         $   util     Δwall      Δ$  kernel
--------------------------------------------------------------------------------------------------------
baseline                             272   1,046,250    2234s     3.477   97%                    
+ CUDA graphs                        272   1,046,250    2234s     3.477   97%               -2%  
+ prefix caching                     272     115,060    2187s     0.682   97%       -2%    -82%  16_prefix_match.cu
+ ragged batching                    511     115,060    1194s     0.651   95%      -45%    -10%  17_ragged_batch.cu
+ bitset logit mask                  511     115,060    1194s     0.651   95%       -0%     -6%  14_logit_mask.cu
+ cascade attention                1,492     115,060     452s     0.629   87%      -62%     -6%  13_prefix_attention.cu
+ chunked prefill                  1,492     115,060     446s     0.629   88%       -1%     -2%  21_chunked_prefill.cu
+ copy-on

The ordering is the lesson, and it is the same one the agent notebook reaches by a different
route: **prefix caching is worth more than everything else combined, and it is not a kernel** —
it is a hash lookup and a decision not to recompute. Every kernel in the list matters *after*
that.

Now the same ladder for a chat workload. Almost none of the same rungs appear, which is the
whole argument for declaring a workload before taking advice about one.

In [6]:
print(sk.ladder_table(sk.recommend(chat).ladder()))

configuration                      tok/s     prefill     wall         $   util     Δwall      Δ$  kernel
--------------------------------------------------------------------------------------------------------
baseline                           3,262       2,048       4s     0.012  100%                    
+ 2x batch                         4,831       2,048       5s     0.012  100%      +34%     -1%  
+ CUDA graphs                      4,831       2,048       5s     0.012  100%               -1%  
+ speculation                      4,384       2,048       6s     0.012  100%      +10%     -1%  18_spec_verify.cu
+ TP=2                             8,695       2,048       3s     0.012  100%      -49%     -1%  10_collectives.cu


## 4 · Which pairs fight, and over what

Two 2× wins rarely give you 4×. The usual explanation is hand-waving about "diminishing
returns"; the real reason is mechanical, and it is that both levers spend the same scarce
resource. Because each lever *declares* what it spends, the interaction can be predicted before
it is measured.

Read the `why` column. A synergy below 1 with **no** declared conflict is a bug — it means a
lever is lying about what it spends, and that is what this table is for rather than a nicety.

In [7]:
mixed = sk.Stack(chat).with_levers(
    "int4 weights", "fp8 KV cache", "2x batch", "speculation", "TP=2", "KV eviction")
print(sk.interaction_table(mixed.interactions()))

A                         B                               A      B  expect  actual  synergy   why
-------------------------------------------------------------------------------------------------
int4 weights              speculation                 1.95x  1.19x   2.32x   1.32x     0.57   both spend spare_compute
2x batch                  speculation                 1.48x  1.19x   1.77x   1.34x     0.76   both spend spare_compute
int4 weights              2x batch                    1.95x  1.48x   2.89x   2.20x     0.76   both spend spare_compute
fp8 KV cache              KV eviction                 1.21x  1.35x   1.64x   1.44x     0.88   both spend accuracy, memory
int4 weights              TP=2                        1.95x  1.99x   3.88x   3.87x     1.00   
speculation               TP=2                        1.19x  1.99x   2.38x   2.37x     1.00   
2x batch                  TP=2                        1.48x  1.99x   2.95x   2.94x     1.00   
TP=2                      KV eviction   

That check earned its place on the first run. `int4 weights` and `fp8 weights` originally
declared that they spend only `accuracy` — and two unexplained 0.57 synergies against
speculation revealed the omission: **shrinking the weight read is what removes the
memory-bound-ness**, and memory-bound-ness is exactly the resource batching and speculation are
paid out of. Both declarations were wrong, the measurement disagreed with them, and the
disagreement was the signal.

`python -m servingkit check` runs that comparison as a gate, so a lever added later cannot
quietly misdeclare itself.

In [8]:
# Inverting the question: what can this workload NOT use, and what would unlock it?
# This is the more useful list if you are deciding what to *build* rather than what to enable.
for lv, missing in sk.Stack(chat).unavailable()[:8]:
    k = f"  [{lv.kernel}]" if lv.kernel else ""
    print(f"  {lv.name:<26} needs {', '.join(missing)}{k}")
print("\nThat is a list of properties worth engineering into existence. Give a chat product")
print("prompt caching and 'repeated_context' appears, which unlocks the largest lever there is.")

  prefix caching             needs repeated_context  [16_prefix_match.cu]
  cascade attention          needs shared_prefix  [13_prefix_attention.cu]
  ragged batching            needs ragged_batch  [17_ragged_batch.cu]
  chunked prefill            needs mid_batch_prefill  [21_chunked_prefill.cu]
  bitset logit mask          needs structured_output  [14_logit_mask.cu]
  grammar on device          needs structured_output  [20_grammar_advance.cu]
  copy-on-write forking      needs forks  [15_kv_fork.cu]
  agent-aware eviction       needs idle_holds, repeated_context  [22_kv_evict.cu]

That is a list of properties worth engineering into existence. Give a chat product
prompt caching and 'repeated_context' appears, which unlocks the largest lever there is.


## 5 · Run the kernel behind a lever

Every lever that names a kernel names a file that compiles and runs. No GPU required — the
shim in `kernelbench/` builds the same `.cu` with `g++` and runs one thread per CUDA thread,
with real barriers. It reports correctness and, deliberately, no timings at all: it emulates
nothing about performance, so printing numbers would be inviting them to be quoted.

In [9]:
lever = sk.LEVERS["cascade attention"]
print(f"{lever.name} -> kernels/{lever.kernel}\n")
print(lever.note, "\n")

r = sk.run_kernel(lever.kernel)
print(f"{r.name} on {r.device}")
print(r.table())
print(f"\nall variants correct: {r.ok}")
print(f"timings meaningful: {r.timed}  (False on a CPU, and the numbers are absent rather")
print("than fabricated — which is the only honest thing to do with an emulator)")

cascade attention -> kernels/13_prefix_attention.cu

Read a shared prefix once per step instead of N times. The online-softmax merge from 06, pointed at a different partition of the keys. The saving tends to (P+S)/S as the fan-out grows, so its factor is the workload's own ratio, not a constant. 



13_prefix_attention on CPU (cuda_shim emulation)
variant                          max err    ok
----------------------------------------------
1 independent per sequence      5.06e-06    ok
2 cascade (prefix read once)    6.27e-06    ok

all variants correct: True
timings meaningful: False  (False on a CPU, and the numbers are absent rather
than fabricated — which is the only honest thing to do with an emulator)


In [10]:
# What that lever is worth at this workload's own shape, from the same arithmetic the kernel
# prints at the bottom of its output.
c = sk.cascade_traffic(prefix_tokens=agent.ctx - agent.output_tokens,
                       suffix_tokens=agent.output_tokens, branches=agent.fanout,
                       kv_bytes_per_token=sk.kv_bytes_per_token(sk.MODELS[agent.model]))
print(f"fan-out {agent.fanout}, prefix {agent.ctx - agent.output_tokens:,} tokens:")
print(f"  independent  {c['independent_bytes']/1e6:8.1f} MB per decode step")
print(f"  cascade      {c['cascade_bytes']/1e6:8.1f} MB   ({c['ratio']:.1f}x less)")
print(f"  the limit as fan-out grows: {c['limit']:.1f}x")

fan-out 4, prefix 65,900 tokens:
  independent   34629.2 MB per decode step
  cascade        8716.3 MB   (4.0x less)
  the limit as fan-out grows: 440.3x


## 6 · Does any of it still agree with the notebooks?

Every model in the package was lifted verbatim from a notebook, and the notebook remains where
the reasoning is explained. So the two must not drift. This pulls each function's source back
out of its `.ipynb`, executes it in isolation, and requires it to agree.

It is the same discipline `tools/verify_console.py` applies to the demo pages' JavaScript. Three
copies of every model — notebook, package, browser — now have to say the same thing, and the
only way to change a number is to change it in one place and watch two checks fail.

In [11]:
import servingkit.selfcheck as selfcheck
selfcheck.run_checks(REPO)

kv                       128 comparisons   ok
speculation            1,440 comparisons   ok
decode step            1,008 comparisons   ok
serving                4,863 comparisons   ok
training               4,608 comparisons   ok
agents                 3,845 comparisons   ok
lever declarations        41 comparisons   ok
                      15,933 total

the package, the notebooks and the levers' own declarations all agree


0

## 7 · The whole plan, for your own numbers

Change the arguments and re-run. The refusals will change with them, which is the part worth
watching: they tell you which of the other forty-nine notebooks is the one to open.

In [12]:
mine = sk.Workload.agent(
    model="Llama-3.1-70B",
    turns=15,
    tool_result_tokens=1200,
    tool_latency_s=1.5,
    fanout=1,               # no sub-agents: watch cascade attention and forking disappear
    batch=16,
    replayed=False,
)
print(sk.plan_report(sk.recommend(mine, gpu="H100 SXM")))

servingkit plan — agent on H100 SXM
agent: Llama-3.1-70B, ctx 22,600, batch 16
  properties: idle_holds, long_context, mid_batch_prefill, predictable_output, ragged_batch, repeated_context, spare_compute, structured_output, weight_bound

levers pulled
  + 2x batch
  + CUDA graphs
  + prefix caching  [16_prefix_match.cu]
  + ragged batching  [17_ragged_batch.cu]
  + bitset logit mask  [14_logit_mask.cu]
  + chunked prefill  [21_chunked_prefill.cu]
  + agent-aware eviction  [22_kv_evict.cu]
  + speculation  [18_spec_verify.cu]
  + grammar on device  [20_grammar_advance.cu]
  + TP=2  [10_collectives.cu]

not applied
  - cascade attention: workload lacks a required property (shared_prefix)
  - copy-on-write forking: workload lacks a required property (forks)
  - batch-invariant reduction: workload lacks a required property (replayed)

the step
  166.104 ms · 588 tok/s · 54.4 ms/token
  bottleneck: attention (memory-bound GEMM), 32% of math capacity idle
  vs baseline: 3.82x

the run
  pref

In [13]:
# And the deployment question, which is a different question: how many GPUs, what does a
# million tokens cost, and what is the cheapest thing that is still fast enough.
rows = sk.cheapest("Llama-3.1-8B", batch=32, ctx=4096, min_tpot_ms=40)
head = f"{'gpu':<12}{'prec':>6}{'tp':>4}{'tok/s':>10}{'tpot ms':>9}{'$/Mtok':>9}   bound"
print(head); print("-" * len(head))
for r in rows[:8]:
    print(f"{r['gpu']:<12}{r['precision']:>6}{r['tp']:>4}{r['decode_tps']:>10,.0f}"
          f"{r['tpot_ms']:>9.1f}{r['cpm']:>9.3f}   {r['bound']}")
print("\nSorted by cost, not speed. The fastest configuration is almost never the cheapest,")
print("and which you want depends on whether you are latency-bound or margin-bound.")

gpu           prec  tp     tok/s  tpot ms   $/Mtok   bound
----------------------------------------------------------
T4            int4   2     3,624      8.8    0.054   memory
A100 80GB     int4   1     9,009      3.6    0.068   memory
MI300X        int4   1    14,301      2.2    0.068   memory
T4            fp16   4     5,598      5.7    0.069   memory
H100 SXM      int4   1    12,925      2.5    0.075   memory
MI325X        int4   1    15,460      2.1    0.075   memory
H200 SXM      int4   1    16,247      2.0    0.077   memory
MI300X         fp8   1    11,377      2.8    0.085   memory

Sorted by cost, not speed. The fastest configuration is almost never the cheapest,
and which you want depends on whether you are latency-bound or margin-bound.


## Where to go next

The plan above named the levers that matter for your workload. Each one has a notebook that
explains why it works and a kernel that implements it:

| if the plan pointed at | read | the kernel |
|---|---|---|
| prefix caching | [Agent Workloads on the Metal](./Agent_Workloads_On_The_Metal.ipynb) · [RAG & Agent Serving](./RAG_Agent_Serving_Patterns.ipynb) | `16_prefix_match.cu` |
| cascade attention, ragged batching | [Attention Kernels From Scratch](./Attention_Kernels_From_Scratch.ipynb) | `13`, `17` |
| chunked prefill, eviction | [vLLM High-Throughput Serving](./vLLM_High_Throughput_Serving.ipynb) · [Long-Context Serving](./LongContext_KV_Compression_Serving.ipynb) | `21`, `22` |
| speculation | [Speculative Decoding](./Speculative_Decoding_Advanced_Serving.ipynb) | `18_spec_verify.cu` |
| grammar, logit masks | [Structured Output](./Structured_Output_Guided_Decoding.ipynb) | `14`, `20` |
| batch invariance | [Agent Workloads, Part 9](./Agent_Workloads_On_The_Metal.ipynb) | `19_batch_invariant.cu` |
| int4 / fp8 | [Quantized Serving Showdown](./Quantized_Serving_Showdown.ipynb) · [Portable Kernels](./Portable_Kernels_Precision_Matrix.ipynb) | `05`, `09` |
| TP, collectives | [Distributed Serving](./Distributed_MultiReplica_Serving.ipynb) | `10_collectives.cu` |
| nothing — you are compute-bound | [The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) · [Measuring GPU Code Honestly](./Measuring_GPU_Code_Honestly.ipynb) | `01`–`03` |

And three things that are not levers but decide whether any of this is true:

- [Measuring GPU Code Honestly](./Measuring_GPU_Code_Honestly.ipynb) — every number here is a
  spec sheet times a fudge factor. This is how to replace them with your own, and
  `sk.recalibrate()` is where to put them.
- [The Optimization Stack](./The_Optimization_Stack.ipynb) — the notebook the interaction matrix
  came from, with the reasoning the table only summarizes.
- [`kernels/README.md`](https://github.com/sugeerth/gpu-training-notebooks/tree/main/kernels) —
  twenty-two programs, in reading order, each introducing the mechanism the next one relies on.

### One honest caveat

`servingkit` is a planning library. Its catalogs are vendor specifications and list prices, and
its efficiency constants are what a *good* kernel reaches. Nothing in it has been measured on
your hardware, and the kernels it points at have been verified for correctness but never timed
on a real GPU. The shape of the answers does not move when you substitute measurements. The
crossovers do — and the crossovers are the part you were going to make a decision on.